# Time Travel and Data Recovery

In [ ]:
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import *

MINIO_ACCESS_KEY = "pkIeKAn4xpjoOgXiHQPw"
MINIO_SECRET_KEY = "JZRBfRszxLZzPeaAQaEXk32JxUKv25DVjUoO06Rk"
DATABASE = "default"
BUCKET_BRONZE = "bronze"
BUCKET_SILVER = "silver"
BUCKET_GOLD = "gold"

In [ ]:
%%time
spark = SparkSession.builder.master("spark://spark-master:7077") \
    .appName("MyAppM5Class01") \
    .config("spark.eventLog.enabled", "true") \
    .config("spark.eventLog.dir", "file:/tmp/spark-logs") \
    .config("spark.history.fs.logDirectory", "file:/tmp/spark-logs") \
    .config("log4j.rootCategory", "INFO, console") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.executor.instances", "4") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "1536m") \
    .config("spark.driver.memory", "1536m") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.storage.memoryFraction", "0.4") \
    .config("spark.shuffle.memoryFraction", "0.5") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "512m") \
    .config("spark.sql.parquet.compression.codec", "gzip") \
    .config("spark.sql.orc.compression.codec", "zlib") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.executor.extraJavaOptions", "-XX:+UseG1GC") \
    .config("spark.cleaner.referenceTracking.cleanCheckpoints", "true") \
    .config("spark.executor.cleanupOnShutdown", "true") \
    .getOrCreate()

In [ ]:
sc = spark.sparkContext
hadoop_conf = sc._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", MINIO_ACCESS_KEY)
hadoop_conf.set("fs.s3a.secret.key", MINIO_SECRET_KEY)
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

## PROJECT 1: HOTEL BOOKING

- [Data source](https://www.kaggle.com/datasets/mojtaba142/hotel-booking)

Let's explore Repairing, Restoring, and Replacing Table Data

1. Reading data from raw from **Staging**
2. Write data into bronze
3. Operation: **Delete** data and cause headaches for ourselves
4. Using the extreme strategy as an alternative, replacing the data through the option: **ReplaceWhere**
5. Operation: **Delete** AGAIN...
6. Time travel
7. **Restoring** our table and saving our job 😃


### 1 - Reading data from raw from **Staging**

In [ ]:
def build_arrival_partition_date(df):
    df = df.withColumn(
        "arrival_date_day_of_month_padded",
        F.lpad("arrival_date_day_of_month", 2, "0")
    )

    df = df.withColumn(
        "arrival_date_month_number",
        F.when(F.col("arrival_date_month") == "January", "01")
        .when(F.col("arrival_date_month") == "February", "02")
        .when(F.col("arrival_date_month") == "March", "03")
        .when(F.col("arrival_date_month") == "April", "04")
        .when(F.col("arrival_date_month") == "May", "05")
        .when(F.col("arrival_date_month") == "June", "06")
        .when(F.col("arrival_date_month") == "July", "07")
        .when(F.col("arrival_date_month") == "August", "08")
        .when(F.col("arrival_date_month") == "September", "09")
        .when(F.col("arrival_date_month") == "October", "10")
        .when(F.col("arrival_date_month") == "November", "11")
        .when(F.col("arrival_date_month") == "December", "12")
    )
    df = (
        df.withColumn("arrival_partition_date", 
          F.to_date(
            F.concat_ws(
                "-", 
                df.arrival_date_year, 
                df.arrival_date_month_number,
                df.arrival_date_day_of_month_padded
            ), 
              "yyyy-MM-dd"
          )
        )
    )
    
    return df.drop("arrival_date_day_of_month_padded", "arrival_date_month_number")

In [ ]:
location_raw = f"s3a://staging"
file = "hotel_booking.csv"
data_origen = f"{location_raw}/{file}"

In [ ]:
df = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(data_origen)
df = df.withColumnRenamed("phone-number", "phone_number")
df = build_arrival_partition_date(df)

In [ ]:
df.select(
    "arrival_date_year",
    "arrival_date_month",
    "arrival_date_day_of_month",
    "arrival_partition_date"
).limit(10).show(truncate=False)

In [ ]:
df.printSchema()

### 2 - Write data into bronze

In [ ]:
table_bronze = "hotel_booking_bronze_datarecovery"
location_bronze = f"s3a://{BUCKET_BRONZE}/delta/{table_bronze}"

In [ ]:
%%time
(
    df.write.format("delta")
    .mode("overwrite")
    .partitionBy("arrival_partition_date")
    .save(location_bronze)
)

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE}.{table_bronze}
    USING DELTA
    LOCATION '{location_bronze}'
""")

### 3 - Operation: **Delete** data and cause headaches for ourselves

In [ ]:
spark.sql(f"""
    DELETE FROM {DATABASE}.{table_bronze} 
    WHERE 
        arrival_partition_date >= '2015-07-01' 
        AND arrival_partition_date <= '2015-07-31'
""")

In [ ]:
# Create a DeltaTable object
delta_table = DeltaTable.forName(spark, f"{DATABASE}.{table_bronze}")
# Get history of table
history_df = delta_table.history()

In [ ]:
(
    history_df.where(F.col("operation") == "DELETE")
    .select(
        "version", "timestamp", "operation",
        "operationMetrics.numRemovedFiles",
        "operationMetrics.numAddedFiles"
    ).show(truncate=False)
)

In [ ]:
spark.sql(f"""
    SELECT
        hotel,
        email,
        lead_time,
        customer_type,
        country,
        arrival_partition_date
    FROM {DATABASE}.{table_bronze}
    WHERE 
        arrival_partition_date >= '2015-07-01' 
        AND arrival_partition_date <= '2015-07-31'
    ORDER BY
        arrival_partition_date
""").limit(10).show(truncate=False)

### 4 - Using the extreme strategy as an alternative, replacing the data through the option: **ReplaceWhere**

In [ ]:
df_replacewhere = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(data_origen)
df_replacewhere = df_replacewhere.withColumnRenamed("phone-number", "phone_number")
df_replacewhere = build_arrival_partition_date(df_replacewhere)

In [ ]:
%%time
condition = "arrival_partition_date >= '2015-07-01' AND arrival_partition_date <= '2015-07-31'"
(
    df_replacewhere
    .where(condition)
    .write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", condition)
    .save(location_bronze)
)

In [ ]:
# Create a DeltaTable object
delta_table = DeltaTable.forName(spark, f"{DATABASE}.{table_bronze}")
# Get history of table
history_df = delta_table.history()
history_df.select("version", "timestamp", "operation").show(truncate=False)

In [ ]:
spark.sql(f"""
    SELECT
        hotel,
        email,
        lead_time,
        customer_type,
        country,
        arrival_partition_date
    FROM {DATABASE}.{table_bronze}
    WHERE 
        arrival_partition_date >= '2015-07-01' 
        AND arrival_partition_date <= '2015-07-31'
    ORDER BY
        arrival_partition_date
""").limit(10).show(truncate=False)

### 5 - Operation: **Delete** AGAIN...

In [ ]:
spark.sql(f"""
    DELETE FROM {DATABASE}.{table_bronze} 
    WHERE 
        arrival_partition_date >= '2015-07-01' 
        AND arrival_partition_date <= '2015-07-31'
""")

In [ ]:
spark.sql(f"""
    SELECT
        hotel,
        email,
        lead_time,
        customer_type,
        country,
        arrival_partition_date
    FROM {DATABASE}.{table_bronze}
    WHERE 
        arrival_partition_date >= '2015-07-01' 
        AND arrival_partition_date <= '2015-07-31'
    ORDER BY
        arrival_partition_date
""").limit(10).show(truncate=False)

### 6 - Time travel

In [ ]:
spark.sql(f"""
    SELECT
        hotel,
        name,
        email,
        lead_time,
        customer_type,
        country,
        arrival_partition_date
    FROM {DATABASE}.{table_bronze} VERSION AS OF 1
    WHERE 
        arrival_partition_date > '2015-07-01' 
        AND arrival_partition_date < '2015-07-31'
    ORDER BY
        arrival_partition_date
""").limit(10).show(truncate=False)

### 7 - Restoring our table and saving our job 😃

In [ ]:
%%time
spark.sql(f"RESTORE TABLE {DATABASE}.{table_bronze} TO VERSION AS OF 0")

In [ ]:
spark.sql(f"""
    SELECT
        hotel,
        name,
        email,
        lead_time,
        customer_type,
        country,
        arrival_partition_date
    FROM {DATABASE}.{table_bronze}
    WHERE 
        arrival_partition_date > '2015-07-01' 
        AND arrival_partition_date < '2015-07-31'
    ORDER BY
        arrival_partition_date
""").limit(10).show(truncate=False)

In [ ]:
# Create a DeltaTable object
delta_table = DeltaTable.forName(spark, f"{DATABASE}.{table_bronze}")
# Get history of table
history_df = delta_table.history()
history_df.select("version", "timestamp", "operation").show(truncate=False)

In [ ]:
spark.stop()